# NINA Output CSV for Spectra
## Adjust Celestial Coordinates for Assigned targets


In [ ]:
!mkdir -p data_folder
notebook_version = "_v9"

# INITIALIZE CELESTIAL COORDINATES PROCESS

In [ ]:
#pip install --upgrade astropy astroquery googlesearch-python wikipedia pandas


In [ ]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki
from IPython.display import Markdown as md

In [ ]:
import requests
import webbrowser
from bs4 import BeautifulSoup

In [ ]:
from datetime import datetime
readable_ts = datetime.now().strftime('%Y%m%d%H%M%S')
# Result: '2024-04-25 10:30:45'
print(f'readable timestamp is {readable_ts}')

In [ ]:
# ra_dec_offset_v5 and output files naming
notebook_name = "nina_targets_for_spectra"

data_folder_name = "data_folder"
output_csvfilename = f"{notebook_name}_{readable_ts}.csv"
print(f"{notebook_name} version is: {notebook_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

In [ ]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")

# Set global offset for spectra in frame to include zero-order
global_offset_arcmin = -3.5 # reduced from -3.0
print(f"Global offset RA & Dec arcmin by {global_offset_arcmin} * sin(camera rotation angle)")

In [ ]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

In [ ]:
# set target default name
target_default_name = "HD"

In [ ]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [ ]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    #original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    print(f'\nsky_PA: {sky_PA}')
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    
    print(f'\noffset_dist: {offset_dist}')
    
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)
    
    print(f'\ndx: {dx} dy: {dy}')
          

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)
    
    print(f'\noffset_frame = {offset_frame}')

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    print(f'\noffset_coord = {offset_coord}')
    print(f'\nnew_coord_ = {new_coord_}')
    
    return new_coord_



In [ ]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
#query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

#results = make_api_request(query)
#if results:
#    print("Data received:", results)


In [ ]:
def googlesearchurl(tname):
    try:
        from googlesearch import search
    except ImportError:
        print("No module named 'google' found")
    #
    # to search
    query = f'{tname} Wikipedia Astronomy'
    
    for j in search(query, tld="co.us", num=10, stop=10, pause=2):
            print(j)
    #for url in search(query, num_results=10, lang="en", unique=True):
    #    print(url)
#
# example
#
#tname = "RR Lyrae"
#googlesearchurl(tname)

In [ ]:
def wikisearchurl(tname):
    # Set language to English (optional)
    wiki.set_lang("en")
    
    # Search for a topic
    results = wiki.search(tname)
    print("Search Results:", results)
    
    # Get a summary of the first result
    summary = wiki.summary(results[0], sentences=1)
    print("\nSummary:", summary)
    
    # Get the full page object
    page = wiki.page(results[0])
    print("\nPage Title:", page.title)
    print("Page URL:", page.url)
#
# example
#
#tname = "RR Lyrae"
#wikisearchurl(tname)

In [ ]:
def obtain_info_for_adhoc_star(tname, ara, adec):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(tname)
    original_coord = SkyCoord(ara, adec, frame='icrs', unit='deg')
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord)

In [ ]:
def obtain_info_for_star(tname):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    original_coord = SkyCoord.from_name(tname)
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord, original_coord)

In [ ]:
def wikisearchurl(tname):
    # 1. Define the Wikipedia URL
    #url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
    url = f"https://en.wikipedia.org/w/index.php?search={tname}&title=Special%3ASearch&fulltext=1&ns0=1"

    # 2. Fetch the page with a User-Agent to avoid 403 Forbidden errors
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    # 3. Check for successful request and parse HTML
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')

        # 4. Find all <a> tags with an 'href' attribute
        # Filters: starts with '/wiki/' and does not contain a ':'
        wiki_links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.startswith('/wiki/') and ':' not in href:
                full_url = f"https://en.wikipedia.org{href}"
                wiki_links.append(full_url)

        # 5. Display the first 10 extracted URLs
        print(*wiki_links[:10], sep="\n")
        print("\nExamine these results above and click any related links ...\n")
        print("\nWhen all else has failed Open Google Search by executing next cell!")


def wikihdsearch(tname):
    # 1. Define the Wikipedia URL
    #url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
    url = f"https://en.wikipedia.org/wiki/{tname}"

    # 2. Fetch the page with a User-Agent to avoid 403 Forbidden errors
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)

    # 3. Check for successful request and parse HTML
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')

        # 4. Find all <a> tags with an 'href' attribute
        # Filters: starts with '/wiki/' and does not contain a ':'
        wiki_links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.startswith('/wiki/Main_Page') and ':' not in href:
                full_url = f"https://en.wikipedia.org{href}"
                wiki_links.append(full_url)

        # 5. Display the first 10 extracted URLs
        print(*wiki_links[:10], sep="\n")
        print("\nExamine these results above and click any related links ...\n")
        print("\nIf star not found try a broader wiki search by executing next cell!")


In [ ]:
def obtain_adjcoord_for_planet(pname, pcoord):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(target_name)
    
    print(pcoord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(pcoord.ra, pcoord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {target_name} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {pcoord.ra.hour:.4f}, {pcoord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {pcoord.ra.deg:.4f}, {pcoord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(pcoord.ra.deg-new_coord.ra.deg):.4f}, \
          {(pcoord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");googlesearchurl(target_name);print("\n")

    return(new_coord, orig_coord)

In [ ]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

## Create a single NINA target Star for imaging

In [ ]:
itn = "Vega"
target_name = input(f"Enter Target Star Name [{itn}]: ") or f"{itn}"
print(f"The Target Star name is: {target_name}")
%run ./sub_nina_targets_for_spectra.ipynb

## Create a Multiple NINA target Stars for imaging

In [ ]:
for skyobj in \
'Zosma', 'Alioth', \
'Denebola', 'Dubhe', \
'Cor Caroli', 'Mizar', \
'Merak', 'Eta Leonis', \
'Phecda', 'Megrez', \
'Tania Australis', 'Minelauva', \
'Seginus':
    target_name = f'{skyobj}'
    %run ./sub_nina_targets_for_spectra.ipynb